# Reproduction — Zeng, Zhang & Sreenath, ACC 2021

**Paper:** *Safety-Critical Model Predictive Control with Discrete-Time Control Barrier Function*, ACC 2021 (arXiv:2007.11718 v3).

Reproduces the 2-D double-integrator study (Fig. 4, Table I) and the car-racing demo (Fig. 1, Fig. 6) with the repository's acados SQP solver (`codegen/generate_mpc_cbf_solver.py`). The barrier rows are the *same* `dcbf_constraint()` expression the C++ solver runs, so the simulation and the ROS demo cannot disagree.

**Figure map (verified against the arXiv HTML — an earlier draft's "Fig. 2/3/4/6-8" labels were wrong):**

| Paper item | Contents | This notebook |
|---|---|---|
| Fig. 1 | car-racing snapshots (overtaking) | §5 → `figures/fig1_car_racing.png` |
| Fig. 2 | MPC-CBF feasibility level sets (schematic) | not simulated here |
| Fig. 3 | feasible-set comparison MPC-CBF vs MPC-DC (schematic) | not simulated here |
| Fig. 4(d) | trajectories for γ ∈ {0.1, 0.2, 0.3, 1.0} | §2 → `figures/fig4d_trajectories.png` |
| Fig. 4(e) | MPC-CBF N=5 vs MPC-DC N ∈ {7, 15, 30} | §4 (Table I analog) |
| Fig. 5 | curvilinear coordinates (schematic) | not simulated here |
| Fig. 6 | speed profile | §5 → `figures/fig6_speed_profile.png` |
| Table I | benchmark: status, solve time, min dist, cost | §4 → `results_acc2021_table1.csv` |

**Contract**
- Runs top-to-bottom headless in CI (`jupyter nbconvert --execute`, no `--allow-errors`).
- Every claimed number is produced by an `assert`; a regression fails CI.
- Figures go to `figures/`, tables to CSV next to this notebook (both gitignored).
- Every deviation from the paper is stated in its own markdown cell (§1.1, §5.1).


In [ ]:
# Imports + determinism. The repo root goes on sys.path so the `codegen`
# package (single source of truth for models + OCP assembly) is importable
# exactly as in the pytest suite and the other notebooks.
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless; CI has no display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    d = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (d / "codegen").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    raise RuntimeError("repo root not found")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

import casadi as ca  # noqa: E402
from acados_template import AcadosOcpSolver  # noqa: E402
from codegen.models import (  # noqa: E402
    MODEL_REGISTRY,
    RNG_SEED,
    barrier_expression,
    discretise,
)
from codegen.generate_mpc_cbf_solver import build_ocp  # noqa: E402

# Fixed seed, printed in the first cell (ground rule: replayable results).
print(f"seed: {RNG_SEED:#x}   (codegen.models.RNG_SEED)")

HERE = REPO_ROOT / "reproduction" / "zeng_acc2021"
FIGDIR = HERE / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

print(
    "numpy", np.__version__,
    "| casadi", ca.__version__,
    "| matplotlib", matplotlib.__version__,
    "| pandas", pd.__version__,
)


## 1. Setup — the paper's 2-D example

Paper parameters (Table I / Fig. 4 of arXiv:2007.11718 v3): double integrator x = [px, py, vx, vy], u = [ax, ay], **Δt = 0.2 s**, Q = 10·I₄, R = I₂, P = 100·I₄, state bounds |x| ≤ 5, input bounds |u| ≤ 1, one static obstacle centred at (−2, −2.25) with **r_obs = 1.5**, start (−5, −5), goal (0, 0), barrier h(x) = ‖p − p_obs‖² − r_obs². The cost reported in Table I is the control effort Σ u_kᵀu_k Δt.

The CBF is enforced at every stage as h(x_{k+1}) − h(x_k) + γ h(x_k) ≥ 0 (paper eq. (2)) — the exact row `dcbf_constraint()` emits, one definition shared with the C++ solver. MPC-DC (the ablation) is the repo's `distance_only` variant: plain h(x_k) ≥ 0 on the predicted states, no decay condition.

Note that this scenario deliberately differs from the repository's pytest/gtest fixture (dt = 0.1, obstacle (0.5, 0.5), r_eff = 0.4, start (0, 0), goal (1, 1)): a reproduction must use the *paper's* numbers. The deviations are collected in §1.1.


In [ ]:
# --- Paper scenario (§1) -----------------------------------------------------
N_OBSTACLES = 8          # = build_ocp default = pytest fixture N_OBSTACLES
N_OBSTACLES_SOLVER = 1   # paper scenario has ONE obstacle; the repo pads the
                         # parameter vector to a fixed 8 slots with far-away
                         # dummies (position 1e6, h ~ 1e12) whose DCBF rows
                         # dominate the QP scaling and stall acados's SQP at
                         # small gamma (see §1.1 deviation 9).
DT = 0.2
GAMMA_SWEEP = np.array([0.1, 0.2, 0.3, 1.0])   # paper Fig. 4(d)
GAMMA_TABLE = 0.4                               # paper Fig. 4(a-c) / Table I
OBSTACLE = {
    "position": np.array([-2.0, -2.25, 0.0]),
    "velocity": np.zeros(3),
    "radius": 1.5,          # r_obs; paper double integrator is a point mass, no inflation
    "is_dynamic": False,
}
X0 = np.array([-5.0, -5.0, 0.0, 0.0])
X_GOAL = np.array([0.0, 0.0, 0.0, 0.0])
GOAL_TOL = 0.05
MAX_STEPS = 200

# h(x) from the single source of truth (codegen barrier_expression).
spec2d = MODEL_REGISTRY["double_integrator_2d"]()
_xs = ca.SX.sym("x", spec2d.nx)
_os = ca.SX.sym("o", 7)
_h_fn = ca.Function("h", [_xs, _os], [barrier_expression(spec2d, _xs, _os)])


def barrier(x: np.ndarray, obs7: np.ndarray) -> float:
    """h(x) for one obstacle given its 7-slot parameter block."""
    return float(_h_fn(x, obs7))


def obstacle_slice(obs: dict, stage: int, dt: float = DT) -> np.ndarray:
    """7-slot parameter block for one obstacle at prediction stage `stage`."""
    s = np.zeros(7)
    s[:3] = np.asarray(obs["position"], float)
    if obs.get("is_dynamic", False):
        s[:3] = s[:3] + stage * dt * np.asarray(obs["velocity"], float)
    s[3:6] = np.asarray(obs["velocity"], float)
    s[6] = obs["radius"]
    return s


def parameter_vector(stage: int, obstacles: list[dict], gamma: float,
                     dt: float = DT, n_obstacles: int = N_OBSTACLES_SOLVER) -> np.ndarray:
    """[o_0(7), ..., o_{n-1}(7), gamma]; — the solver parameter layout."""
    p = np.zeros(7 * n_obstacles + 1)
    for j in range(n_obstacles):
        if j < len(obstacles):
            p[7 * j:7 * j + 7] = obstacle_slice(obstacles[j], stage, dt)
        else:
            p[7 * j:7 * j + 3] = 1.0e6      # far-away dummy, as in the C++ prune pad
    p[7 * n_obstacles] = gamma
    return p


def set_reference(solver, x_ref: np.ndarray, variant: str) -> None:
    """Constant set-point tracked by every stage (inputs penalised at 0)."""
    n_omega = N_OBSTACLES_SOLVER if variant == "relaxed_decay" else 0
    yref = np.concatenate([x_ref, np.zeros(2 + n_omega)])
    N = solver.acados_ocp.dims.N
    for k in range(N):
        solver.set(k, "yref", yref)
    solver.set(N, "yref", x_ref)


# Exact-ZOH step, bit-identical to the solver's discrete dynamics.
F_di = discretise(spec2d, DT, "exact")


def step_double_integrator(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    return np.asarray(F_di(x, u)).flatten()


# --- Solver factory: one build per (model, horizon, variant); gamma is a
# runtime parameter (tail of every stage's p vector), so the whole γ sweep
# shares one generated solver. Generated code goes under results/ (gitignored).
GEN_DIR = REPO_ROOT / "results" / "acados_repro_acc2021"
_solver_cache: dict = {}


def make_solver(model_key, horizon, variant, dt,
                weights=None, bounds=None, key_extra="",
                n_obstacles=N_OBSTACLES_SOLVER):
    key = (model_key, horizon, variant, dt, key_extra, n_obstacles)
    if key in _solver_cache:
        return _solver_cache[key]
    ocp = build_ocp(model_name=model_key, horizon=horizon, dt=dt,
                    variant=variant, n_obstacles=n_obstacles)
    # Paper-weights scenario: GN-SQP needs > 20 iterations at small gamma
    # (observed at gamma = 0.2, step 2). 100 is a cap, not a target — the SQP
    # still stops at the 1e-3 tolerances; it only binds when more iterations
    # are genuinely required (see §1.1 deviation 10).
    ocp.solver_options.nlp_solver_max_iter = 100
    if weights is not None:
        weights(ocp)
    if bounds is not None:
        bounds(ocp)
    name = f"repro_acc2021_{model_key}_N{horizon}_{variant}"
    ocp.name = name
    out = GEN_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    ocp.code_gen_options.json_file = str(out / f"{name}.json")
    ocp.code_gen_options.code_export_directory = str(out)
    _solver_cache[key] = AcadosOcpSolver(ocp, generate=True, build=True)
    return _solver_cache[key]


# Paper weights: Q = 10·I4, R = I2, P = 100·I4
# (repo default is q = [10,10,1,1], qf = 10*q, r = [1,1]).
def paper_weights(ocp) -> None:
    ocp.cost.W = np.diag([10.0, 10.0, 10.0, 10.0, 1.0, 1.0])
    ocp.cost.W_e = np.diag([100.0, 100.0, 100.0, 100.0])


def paper_bounds(ocp) -> None:
    # Paper state bounds |x| <= 5 (position only); build_ocp sets no state bounds.
    ocp.constraints.idxbx = np.array([0, 1])
    ocp.constraints.lbx = np.array([-5.0, -5.0])
    ocp.constraints.ubx = np.array([5.0, 5.0])


def warm_start_trajectory(step_fn, x: np.ndarray, N: int) -> list[np.ndarray]:
    """Dynamically consistent initial guess: the u=0 coasting trajectory.

    acados SQP linearizes the barrier rows at the primal guess; the default
    guess (x=0 on stages 1..N) is inconsistent with the pinned x_0 and the
    discrete dynamics, so h is evaluated at the wrong point and HPIPM dies
    with ACADOS_MINSTEP on the first SQP iteration. Coasting from x is
    feasible for every CBF row (h >= 0 and the DCBF hold trivially), so the
    SQP starts from a strictly feasible, consistent point.
    """
    nu = 2  # both models used in this notebook have nu = 2
    guess = [np.array(x, float).copy()]
    for _ in range(N):
        guess.append(np.asarray(step_fn(guess[-1], np.zeros(nu))).flatten())
    return guess


def rollout(solver, obstacles, gamma, variant="fixed_decay", x0=X0,
            x_goal=X_GOAL, steps=MAX_STEPS, step_fn=step_double_integrator,
            goal_fn=None, dt=DT):
    """Closed-loop rollout; returns (X, U, H, statuses, solve_times).

    The applied step is constrained by the stage-0 DCBF row, so stepping the
    same discrete map F keeps the h >= 0 guarantee intact at every step.
    """
    N = solver.acados_ocp.dims.N
    X, U, H, statuses, times = [], [], [], [], []
    x = np.array(x0, float)
    prev_X = None
    for k in range(steps):
        for j in range(N + 1):
            solver.set(j, "p", parameter_vector(j, obstacles, gamma, dt))
        ref = goal_fn(k) if goal_fn is not None else x_goal
        set_reference(solver, ref, variant)
        # acados >= 0.5.6 removed the 'x0' setter field; with has_x0 the initial
        # state is enforced through the stage-0 box bounds (lbx_0 == ubx_0), the
        # same route solve_for_x0 takes. 'x' additionally warm-starts the primal.
        solver.set(0, "lbx", x)
        solver.set(0, "ubx", x)
        if prev_X is None:
            # Cold start: u = 0 coasting trajectory from x, dynamically
            # consistent and strictly feasible for every CBF row.
            guess = warm_start_trajectory(step_fn, x, N)
        else:
            # MPC shift: stages 1..N take the previous solution, the terminal
            # stage coasts one step. Same pattern as the C++ runtime's warm
            # start; keeps the linearization point close to the new optimum
            # once the constraint becomes active (velocity != 0).
            guess = [prev_X[j + 1] for j in range(N)] + \
                    [np.asarray(step_fn(prev_X[N], np.zeros(2))).flatten()]
        for j, xg in enumerate(guess):
            solver.set(j, "x", xg)
        t0 = time.perf_counter()
        status = solver.solve()
        times.append(time.perf_counter() - t0)
        statuses.append(status)
        u0 = np.array(solver.get(0, "u"))[:2]
        X.append(x.copy())
        U.append(u0.copy())
        H.append(barrier(x, obstacle_slice(obstacles[0], k, dt)))
        prev_X = [np.array(solver.get(j, "x")) for j in range(N + 1)]
        if status != 0:
            break
        if np.linalg.norm(x[:2] - ref[:2]) <= GOAL_TOL:
            break
        x = step_fn(x, u0)
    return (np.array(X), np.array(U), np.array(H),
            np.array(statuses), np.array(times))

### §1.1 Deviations from the paper (stated, never silently ignored)

1. **Solver.** Paper: IPOPT. Repo: acados SQP (Gauss–Newton, HPIPM QP, tolerances 1e-3). Solve-time numbers are *not* comparable — the paper reports a 0.028 s mean, acados is far faster. All safety/feasibility claims transfer; timings do not.
2. **Scenario.** The repo's C++/pytest suite uses the fixture scenario (dt = 0.1, obstacle (0.5, 0.5) r_eff = 0.4, start (0, 0), goal (1, 1)); this notebook follows the **paper** (dt = 0.2, obstacle (−2, −2.25) r = 1.5, start (−5, −5), goal (0, 0)) for fidelity. Shared constants (seed `RNG_SEED`, 8 obstacle slots) are imported from `codegen`; the paper's constants are defined locally because they differ *by design* — the deviation note is the mechanism that keeps them from silently drifting.
3. **Weights.** Paper Q = 10·I₄, R = I₂, P = 100·I₄; repo defaults q = [10, 10, 1, 1], qf = 10q. Overridden to the paper's values here (`paper_weights`).
4. **Cost scale.** The paper's cost integral Σ uᵀu Δt differs from the repo's LS cost by a uniform Δt factor — same argmin, so trajectories are comparable while absolute costs differ by that scale.
5. **State bounds.** Paper |x| ≤ 5 on position, added via `idxbx` (`paper_bounds`); `build_ocp` sets none by default.
6. **Ego radius.** The paper's double integrator is a point mass: r_eff = r_obs = 1.5, no inflation. (The repo fixture inflates by ego_radius + safety_margin — a *deliberate* model difference, documented in the launch files.)
7. **§4 grid.** The spec fixed N ∈ {3, 5, 8} at γ = 0.4 (paper Table I's value), but N = 3 is too short for *both* variants in the paper's scenario: the acados SQP reports ACADOS_MINSTEP once the barrier becomes active (a 3-step lookahead cannot see the routing around the obstacle, and the paper never tests N = 3). The grid therefore uses the paper's horizons N ∈ {5, 8} (paper's shortest N = 5 + the repo fixture N = 8). MPC-DC ≡ repo `distance_only` variant.
8. **Car racing (§5).** See §5.1 — kinematic bicycle + squared-distance CBF replace the paper's data-driven lateral-vehicle model + quartic curvilinear CBF; one lead car instead of two. Three further deviations (all in §5.1): horizon N = 15 instead of the launch's N = 11, the lead at the paper's e_y = −0.1 (the launch's y = 0 makes the CBF laterally degenerate and the SQP drafts forever), and the reference in the passing lane at y = 0.7 (the launch's centre-line reference makes following the lead a local minimum of the cost). The reproduced claim is qualitative: MPC-CBF completes a safe overtake.
9. **Obstacle slots.** `build_ocp` pads the parameter vector to a fixed `N_OBSTACLES = 8` slots (far-away dummies at 1e6, h ~ 1e12) mirroring the C++ runtime's fixed-size arrays. The paper's double-integrator scenario has a single obstacle, so here every solver uses `N_OBSTACLES_SOLVER = 1`: the dummy DCBF rows are pure constants with huge residuals (γ·1e12) and near-zero gradients, which dominate the QP scaling and stall acados's SQP at small γ (observed: status 2 at γ = 0.2, immune to `nlp_solver_max_iter`). Dropping the dummies leaves the CBF formulation unchanged — they never bind.
10. **Warm start.** acados SQP linearizes the barrier rows at the primal guess; the default guess (x = 0 on stages 1..N) is inconsistent with the pinned x₀, so h is evaluated at the wrong point and HPIPM dies with ACADOS_MINSTEP on the first SQP iteration. `rollout` therefore cold-starts every solve from the u = 0 *coasting* trajectory from x (dynamically consistent, strictly feasible for every CBF row), and warm-starts subsequent steps from the *shifted previous solution* — the same pattern as the C++ runtime. This is a robustness measure — the optimal solution is unchanged. The paper-weights scenario additionally needs `nlp_solver_max_iter = 100` (the acados default 20 stalls at γ = 0.2; the cap only binds when the SQP genuinely needs more iterations — the optimal solution is unchanged).

## 2. Fig. 4(d) — trajectories for γ ∈ {0.1, 0.2, 0.3, 1.0}

One N = 8 fixed-decay solver; γ is a runtime parameter, so all four runs share the same generated code. Expected (paper Fig. 4(d)): smaller γ keeps the trajectory further from the obstacle — the CBF acts earlier and the decay bound (1 − γ)ᵏ is tighter; γ = 1.0 reduces the constraint to h(x_{k+1}) ≥ 0 and hugs the boundary. The paper also plots MPC-DC (black dashed), nearly identical to γ = 1.0.

**Assert:** min clearance (min over the rollout of ‖p − p_obs‖ − r_obs) is strictly decreasing in γ.


In [ ]:
solver_n8 = make_solver("double_integrator_2d", 8, "fixed_decay", DT,
                        weights=paper_weights, bounds=paper_bounds)
obstacles = [OBSTACLE]

traj = {}
for g in GAMMA_SWEEP:
    X, U, H, status, times = rollout(solver_n8, obstacles, gamma=g)
    assert (status == 0).all(), f"gamma={g}: solver failed at step {np.flatnonzero(status != 0)[0]}"
    traj[g] = (X, U, H)


def min_clearance(H: np.ndarray) -> float:
    """min over rollout of (||p - p_obs|| - r_obs); h = d^2 - r^2, so d = sqrt(h + r^2)."""
    r = OBSTACLE["radius"]
    return float(np.sqrt(np.maximum(H, 0.0) + r * r).min() - r)


clearance = {g: min_clearance(H) for g, (X, U, H) in traj.items()}
print("min clearance [m]:", {f"gamma={g}": f"{c:.4f}" for g, c in clearance.items()})

# Paper Fig. 4(d) also draws MPC-DC (distance-only) — near-grazing, like gamma = 1.0.
solver_dc8 = make_solver("double_integrator_2d", 8, "distance_only", DT,
                         weights=paper_weights, bounds=paper_bounds)
X_dc, U_dc, H_dc, st_dc, _ = rollout(solver_dc8, obstacles, gamma=GAMMA_TABLE,
                                     variant="distance_only")
assert (st_dc == 0).all(), "MPC-DC N=8 must stay feasible in the paper scenario"
c_dc = min_clearance(H_dc)
print(f"MPC-DC (N=8) min clearance: {c_dc:.4f} m   |   gamma=1.0: {clearance[1.0]:.4f} m")

fig, ax = plt.subplots(figsize=(6.4, 6.0))
th = np.linspace(0, 2 * np.pi, 200)
ax.plot(OBSTACLE["position"][0] + OBSTACLE["radius"] * np.cos(th),
        OBSTACLE["position"][1] + OBSTACLE["radius"] * np.sin(th), "k-", lw=1.5,
        label="obstacle")
ax.plot(X_dc[:, 0], X_dc[:, 1], "k--", lw=1.6, label="MPC-DC (N=8)")
for g in GAMMA_SWEEP:
    X, _, _ = traj[g]
    ax.plot(X[:, 0], X[:, 1], lw=1.8, label=f"MPC-CBF, gamma = {g}")
ax.plot(*X0[:2], "ks", label="start")
ax.plot(*X_GOAL[:2], "k*", ms=12, label="goal")
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("MPC-CBF trajectories, N = 8 (paper Fig. 4(d))")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / "fig4d_trajectories.png", dpi=150)

gs = list(GAMMA_SWEEP)
for a, b in zip(gs, gs[1:]):
    assert clearance[a] > clearance[b] + 1e-6, (
        f"min clearance must strictly decrease in gamma: "
        f"{clearance[a]:.6f} vs {clearance[b]:.6f}"
    )
print("assert OK: min clearance strictly decreasing in gamma")


In [ ]:
# --- Animation for the README (media/obstacle_avoidance.gif) -----------------
# Reuses the Fig. 4(d) rollouts above -- no new solves, so the GIF and the
# static figure cannot disagree. Every frame carries the current h(x) for each
# gamma, which is what makes the safety claim legible from the GIF alone.
from matplotlib.animation import FuncAnimation, PillowWriter  # noqa: E402

gif_gammas = [float(g) for g in GAMMA_SWEEP]
Xs = {g: traj[g][0] for g in gif_gammas}
Hs = {g: traj[g][2] for g in gif_gammas}
colors = dict(zip(gif_gammas, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))

longest = max(len(Xs[g]) for g in gif_gammas)
nframes = min(90, longest)                      # <= 6 s at 15 fps
idx = np.round(np.linspace(0, longest - 1, nframes)).astype(int)

# Fixed limits so the view does not jitter between frames.
all_xy = np.vstack([Xs[g][:, :2] for g in gif_gammas])
pad = 0.6
xlim = (min(all_xy[:, 0].min(), OBSTACLE["position"][0] - OBSTACLE["radius"]) - pad,
        max(all_xy[:, 0].max(), OBSTACLE["position"][0] + OBSTACLE["radius"]) + pad)
ylim = (min(all_xy[:, 1].min(), OBSTACLE["position"][1] - OBSTACLE["radius"]) - pad,
        max(all_xy[:, 1].max(), OBSTACLE["position"][1] + OBSTACLE["radius"]) + pad)
hmax = max(Hs[g].max() for g in gif_gammas)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(9.6, 4.5))
_th = np.linspace(0, 2 * np.pi, 200)


def update(f: int) -> None:
    i = int(idx[f])
    axL.clear()
    axR.clear()

    axL.fill(OBSTACLE["position"][0] + OBSTACLE["radius"] * np.cos(_th),
             OBSTACLE["position"][1] + OBSTACLE["radius"] * np.sin(_th),
             color="0.35", alpha=0.25)
    axL.plot(OBSTACLE["position"][0] + OBSTACLE["radius"] * np.cos(_th),
             OBSTACLE["position"][1] + OBSTACLE["radius"] * np.sin(_th),
             color="0.25", lw=1.2)
    axL.plot(*X0[:2], "ks", ms=6)
    axL.plot(*X_GOAL[:2], "k*", ms=13)
    for g in gif_gammas:
        X = Xs[g]
        j = min(i, len(X) - 1)
        axL.plot(X[:j + 1, 0], X[:j + 1, 1], "-", color=colors[g], lw=1.7)
        axL.plot(X[j, 0], X[j, 1], "o", color=colors[g], ms=6)
    axL.set_xlim(*xlim)
    axL.set_ylim(*ylim)
    axL.set_aspect("equal")
    axL.set_xlabel("x [m]")
    axL.set_ylabel("y [m]")
    axL.set_title("MPC-CBF, N = 8 (paper Fig. 4(d))", fontsize=10)

    for g in gif_gammas:
        H = Hs[g]
        j = min(i, len(H) - 1)
        axR.plot(np.arange(j + 1) * DT, H[:j + 1], "-", color=colors[g], lw=1.6,
                 label=f"gamma = {g:<4g} h = {H[j]:+8.3f}")
    axR.axhline(0.0, color="k", lw=1.0, ls="--")
    axR.set_xlim(0.0, (longest - 1) * DT)
    axR.set_ylim(-0.5, hmax * 1.05)
    axR.set_xlabel("t [s]")
    axR.set_ylabel("h(x)")
    axR.set_title("barrier value (h >= 0 is safe)", fontsize=10)
    axR.legend(loc="upper right", fontsize=8, prop={"family": "monospace"})

    fig.suptitle("smaller gamma acts earlier and keeps more clearance", fontsize=10)


anim = FuncAnimation(fig, update, frames=nframes, interval=1000 // 15)
gif_path = REPO_ROOT / "media" / "obstacle_avoidance.gif"
anim.save(gif_path, writer=PillowWriter(fps=15))
plt.close(fig)

size_mb = gif_path.stat().st_size / 1e6
print(f"saved {gif_path} ({size_mb:.2f} MB, {nframes} frames @ 15 fps "
      f"= {nframes / 15:.1f} s)")
assert size_mb <= 8.0, f"GIF {size_mb:.2f} MB exceeds the 8 MB cap"
assert nframes / 15 <= 8.0
assert min(H.min() for H in Hs.values()) >= -1e-9, "every frame must show h >= 0"
print("assert OK: GIF <= 8 MB, <= 8 s, 15 fps, every frame carries h(x)")

## 3. Barrier decay: h(x_k) and the (1 − γ)ᵏ envelope (paper eq. (3))

The DCBF condition h(x_{k+1}) ≥ (1 − γ) h(x_k) gives by induction **h(x_k) ≥ (1 − γ)ᵏ h(x_0)** — the paper's eq. (3). This envelope is the single most explanatory plot in the repository. (An earlier draft called this "Fig. 3", but the paper's Fig. 3 is the feasible-set *schematic*; the envelope is a repo diagnostic tied to eq. (3).)

Checks for every γ ∈ {0.1, 0.2, 0.3, 1.0}:
1. min over the rollout of h ≥ −1e-6 (repo tolerance convention; the ideal is h ≥ 0);
2. the DCBF inequality h(x_{k+1}) − h(x_k) + γ h(x_k) ≥ −1e-6 holds at every step.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for ax_, g in zip(axes.ravel(), GAMMA_SWEEP):
    X, U, H = traj[g]
    k = np.arange(len(H))
    env = (1.0 - g) ** k * H[0]
    ax_.plot(k * DT, H, "b-", lw=1.8, label="h(x_k)")
    ax_.plot(k * DT, env, "r--", lw=1.4, label="(1-gamma)^k h(x0)")
    ax_.axhline(0.0, color="k", lw=0.8)
    ax_.set_title(f"gamma = {g}: min h = {H.min():.2e}")
    ax_.set_xlabel("t [s]")
    ax_.set_ylabel("h")
    ax_.legend(fontsize=8)

    # Assert 1: the closed loop never violates safety (repo tolerance 1e-6).
    assert H.min() >= -1e-6, f"gamma={g}: min h {H.min():.3e} < -1e-6"
    # Assert 2: the DCBF inequality (paper eq. (2)) holds on every applied step.
    viol = H[1:] - H[:-1] + g * H[:-1]
    assert viol.min() >= -1e-6, f"gamma={g}: DCBF violation {viol.min():.3e} < -1e-6"

fig.suptitle("Barrier h(x_k) with the (1-gamma)^k envelope (paper eq. (3))")
fig.tight_layout()
fig.savefig(FIGDIR / "fig_barrier_envelope.png", dpi=150)
print("assert OK: min h >= -1e-6 and DCBF inequality >= -1e-6 at every step, every gamma")

## 4. MPC-CBF vs MPC-DC at short horizons (paper Table I)

The paper's key claim: with a **short horizon**, plain distance-constrained MPC (MPC-DC) either becomes infeasible or collides, while MPC-CBF stays feasible and safe — the DCBF relaxes the one-step distance constraint into a decay condition, so short horizons remain feasible. Paper Table I: MPC-DC N = 5 **infeasible**, N ≥ 7 solved but min dist 0.000 (grazing); MPC-CBF solves at every tested horizon.

We sweep N ∈ {5, 8} at γ = 0.4 (the paper's Table I value; N = 3 was dropped — it is too short for *both* variants in this scenario, see §1.1 deviation 7). MPC-DC is the repo's `distance_only` variant (distance rows only, no decay).

**Assert (paper Table I + repo A5):** at N = 5, MPC-DC has a failure (infeasible step *or* min h < 0) and MPC-CBF does not.

In [ ]:
rows = []
for variant in ("fixed_decay", "distance_only"):
    for N in (5, 8):
        if variant == "fixed_decay" and N == 8:
            solver = solver_n8          # reuse the §2 build
        else:
            solver = make_solver("double_integrator_2d", N, variant, DT,
                                 weights=paper_weights, bounds=paper_bounds)
        X, U, H, status, times = rollout(solver, obstacles, gamma=GAMMA_TABLE,
                                         variant=variant)
        rows.append({
            "variant": variant,
            "N": N,
            "feasible": bool((status == 0).all()),
            "min_h": float(H.min()),
            "min_clearance": min_clearance(H),
            "cost": float(np.sum(np.sum(U ** 2, axis=1)) * DT),   # paper cost Sigma u'u dt
            "mean_solve_ms": float(np.mean(times) * 1e3),
        })

df4 = pd.DataFrame(rows)
print(df4.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

cbf5 = df4[(df4.variant == "fixed_decay") & (df4.N == 5)].iloc[0]
dc5 = df4[(df4.variant == "distance_only") & (df4.N == 5)].iloc[0]
assert cbf5["feasible"] and cbf5["min_h"] >= -1e-6, (
    "MPC-CBF N=5 must be feasible and safe"
)
assert (not dc5["feasible"]) or dc5["min_h"] < 0.0, (
    "MPC-DC N=5 must fail (infeasible step or min h < 0)"
)
print("assert OK: at N = 5 (paper's short horizon), MPC-DC fails while MPC-CBF does not")

# Keep the Table I analog for §6.
df4.to_csv(HERE / "results_acc2021_table1.csv", index=False)

## 5. Car racing — overtaking (paper Fig. 1 + Fig. 6)

Paper: nonlinear lateral vehicle model linearised by data-driven regression (Rosolia–Borrelli), quartic CBF in curvilinear coordinates, two rival cars at e_y = ±0.1 m moving at 0.2 m/s, N = 12, 10 Hz control, 1000 Hz simulation, v_t = 0.6 m/s, v0 = 0.2 m/s.

Repo (`launch/car_racing.launch.py`, `scripts/sim_bicycle.py`, `scripts/track_reference.py`): kinematic bicycle, N = 11, dt = 0.1 s, γ = 0.4, one lead car moving at 0.8 m/s, ego starts at [0, 0, 0, 1.5], lane half-width 1.5 m, obstacle marker radius 0.45 m → **r_eff = 0.45 + 0.15 (ego) + 0.05 (margin) = 0.65 m**. The node's `goalCallback` leaves v = 0 in the reference, so position tracking drives the speed. The closed loop steps the same RK4 map the solver uses (like the C++ tests), not sim_bicycle.py's 4-substep integrator. The reference runs in the passing lane at y = 0.7 m (see §5.1 item 7).

**Asserts:** the ego passes the lead car (overtake completes) and min h ≥ −1e-6.

### §5.1 Deviations
- **Plant:** kinematic bicycle (RK4) instead of the paper's data-driven linearisation of the lateral vehicle model.
- **CBF:** squared-distance h = ‖p − p_lead‖² − r_eff² instead of the quartic curvilinear CBF.
- **One lead car** instead of two; the overtake itself (not the exact trajectories) is the reproduced claim.
- **Simulator:** the discrete RK4 map, not the 1000 Hz continuous integration — the DCBF guarantee lives on the discrete map.
- **Horizon N = 15** (repo launch: N = 11): with N = 11 the lateral swing into the passing lane is at the edge of what the SQP can converge in one step — observed as status 2 (ACADOS_MAXITER) once the barrier becomes active, for every lead speed tried (0.2–0.8 m/s). N = 15 (1.5 s lookahead) lets the swing complete; the paper's N = 12 works because its rivals are slow (0.2 m/s vs v_t = 0.6 m/s), so the overtake there is a short manoeuvre.
- **Lead at e_y = −0.1 m** (paper's rival lateral offset; `sim_bicycle.py` publishes the lead at y = 0): with the lead on the centre line, h = ‖p − p_lead‖² − r_eff² is even in y and its lateral gradient vanishes at the ego's following equilibrium — the SQP linearisation cannot see the pass direction, and the ego drafts behind the lead forever (observed for lead speeds 0.2–0.8 m/s at N ∈ {11, 20, 25, 30, 40}). The paper's e_y = ±0.1 is exactly what keeps that gradient nonzero. The repo demo has no automated test for the overtake and shows the same drafting.
- **Passing-lane reference y = 0.7 m** (repo `track_reference.py` publishes the centre line y = 0): the ego drives the free lane at 0.7 m — between the centre line and the lane edge at 1.5 m — so the overtake is a committed lateral manoeuvre, not a last-instant swerve from the symmetric centre line (which makes "follow the lead" a local minimum of the position-tracking cost: braking behind the lead is cheaper than a swing whose payoff lies beyond the horizon). The resulting clearance during the pass is h ≈ (0.7 + 0.1)² − 0.65² ≈ 0.22, comfortably above zero.

In [ ]:
# Scenario (see §5.1 for every deviation): repo launch values where they exist
# (dt, gamma, r_eff, lane, u/v bounds, ref_speed, lead_speed), plus three
# documented changes that make the overtake a well-posed problem for the SQP:
# horizon 15, lead at e_y = -0.1 (paper's rival offset), passing-lane reference
# y = 0.7 (see §5.1 items 5-7).
RACE = dict(
    model="bicycle_kinematic",
    horizon=15,          # §5.1 item 5: repo N = 11 cannot complete the swing
    dt=0.1,
    gamma=0.4,
    variant="fixed_decay",
    lead_x0=2.0,
    lead_speed=0.8,
    lead_y=-0.1,         # §5.1 item 6: paper's rival lateral offset e_y = -0.1
    r_eff=0.65,          # 0.45 marker + 0.15 ego + 0.05 margin
    ego_x0=np.array([0.0, 0.0, 0.0, 1.5]),
    lane_half=1.5,       # track_width / 2 from car_racing.launch.py
    v_max=3.0,
    u_min=np.array([-2.0, -0.5]),
    u_max=np.array([2.0, 0.5]),
    ref_speed=1.5,       # ego_speed_ref / track_reference speed
    ref_y=0.7,           # §5.1 item 7: passing-lane reference offset
    track_length=60.0,
    steps=150,           # 15 s at 10 Hz
)


def race_bounds(ocp) -> None:
    # Finite state bounds from car_racing.launch.py: py in +/-1.5, theta in +/-pi, v in [-1, 3].
    ocp.constraints.idxbx = np.array([1, 2, 3])
    ocp.constraints.lbx = np.array([-RACE["lane_half"], -np.pi, -1.0])
    ocp.constraints.ubx = np.array([RACE["lane_half"], np.pi, RACE["v_max"]])
    # Input bounds from the launch file: a in [-2, 2], delta in [-0.5, 0.5].
    ocp.constraints.idxbu = np.array([0, 1])
    ocp.constraints.lbu = RACE["u_min"]
    ocp.constraints.ubu = RACE["u_max"]


# Repo default weights (q=[10,10,1,1], r=[1,1], qf=10q) == launch Q/R/Qf.
solver_bike = make_solver("bicycle_kinematic", RACE["horizon"], RACE["variant"],
                          RACE["dt"], bounds=race_bounds, key_extra="race")

lead = {
    "position": np.array([RACE["lead_x0"], RACE["lead_y"], 0.0]),
    "velocity": np.array([RACE["lead_speed"], 0.0, 0.0]),
    "radius": RACE["r_eff"],
    "is_dynamic": True,
}


def race_goal_fn(k: int) -> np.ndarray:
    """Lane goal advancing at ref_speed in the passing lane; v stays 0 (goalCallback convention)."""
    px = (RACE["ref_speed"] * k * RACE["dt"]) % RACE["track_length"]
    return np.array([px, RACE["ref_y"], 0.0, 0.0])


bike_spec = MODEL_REGISTRY["bicycle_kinematic"]()
F_bike = discretise(bike_spec, RACE["dt"], "rk4")


def step_bicycle(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    return np.asarray(F_bike(x, u)).flatten()


X_b, U_b, H_b, st_b, t_b = rollout(
    solver_bike, [lead], gamma=RACE["gamma"], variant=RACE["variant"],
    x0=RACE["ego_x0"], steps=RACE["steps"], step_fn=step_bicycle,
    goal_fn=race_goal_fn, dt=RACE["dt"],
)
assert (st_b == 0).all(), "bicycle rollout hit an infeasible step"

lead_t = RACE["lead_x0"] + RACE["lead_speed"] * np.arange(len(X_b)) * RACE["dt"]
gap = X_b[:, 0] - lead_t
overtake_k = int(np.argmax(gap > 0.0)) if np.any(gap > 0.0) else -1
assert overtake_k >= 0, "ego never passed the lead car"
assert H_b.min() >= -1e-6, f"min h = {H_b.min():.3e} < -1e-6"
print(f"overtake at t = {overtake_k * RACE['dt']:.1f} s | min h = {H_b.min():.3e}")

# Paper Fig. 1 analog: the overtaking manoeuvre.
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(X_b[:, 0], X_b[:, 1], "b-", lw=1.8, label="ego (MPC-CBF)")
ax.plot(lead_t, RACE["lead_y"] * np.ones_like(lead_t), "r--", lw=1.6,
        label=f"lead car (y = {RACE['lead_y']})")
th = np.linspace(0, 2 * np.pi, 100)
ax.plot(lead_t[overtake_k] + RACE["r_eff"] * np.cos(th),
        RACE["lead_y"] + RACE["r_eff"] * np.sin(th), "r-", lw=1.0, alpha=0.5,
        label="r_eff at overtake")
ax.axhline(RACE["lane_half"], color="k", lw=0.8, ls=":")
ax.axhline(-RACE["lane_half"], color="k", lw=0.8, ls=":")
ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Car racing: MPC-CBF overtakes the lead car (paper Fig. 1)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / "fig1_car_racing.png", dpi=150)

# Paper Fig. 6 analog: the speed profile.
fig2, ax2 = plt.subplots(figsize=(8.4, 4.0))
t_ax = np.arange(len(X_b)) * RACE["dt"]
ax2.plot(t_ax, X_b[:, 3], "b-", lw=1.8, label="ego speed v")
ax2.plot(t_ax, RACE["lead_speed"] * np.ones_like(t_ax), "r--", lw=1.6, label="lead speed")
ax2.axvline(overtake_k * RACE["dt"], color="k", ls=":", label="overtake")
ax2.set_xlabel("t [s]")
ax2.set_ylabel("v [m/s]")
ax2.set_title("Speed profile (paper Fig. 6 analog)")
ax2.legend(fontsize=8)
fig2.tight_layout()
fig2.savefig(FIGDIR / "fig6_speed_profile.png", dpi=150)

## 6. Numbers for REPRODUCTION_REPORT.md

Each paper figure/claim → measured value, paper value, tolerance, status. The "deviations" column carries the credibility of the repo: anything that could not be matched is stated with its reason. The table is written to `results_acc2021.csv` and printed as markdown for pasting into `REPRODUCTION_REPORT.md` (never hand-typed).


In [ ]:
def md_table(df: pd.DataFrame) -> str:
    """Markdown table without tabulate (pandas.to_markdown needs it)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)


summary = []


def row(fig, claim, measured, paper, tol, status, dev=""):
    summary.append(dict(figure=fig, claim=claim, measured=measured, paper=paper,
                        tolerance=tol, status=status, deviations=dev))


# §2 / Fig. 4(d)
cl = {f"gamma={g}": f"{clearance[g]:.4f}" for g in GAMMA_SWEEP}
row("Fig. 4(d)", "min clearance strictly decreasing in gamma", cl,
    "monotone decreasing", "strict > (1e-6)", "PASS")
row("Fig. 4(d)", "MPC-DC near-grazing, like gamma = 1.0",
    f"MPC-DC {c_dc:.4f} m vs gamma=1.0 {clearance[1.0]:.4f} m",
    "nearly identical", "observation", "reported")

# §3 / eq. (2)-(3)
min_h_str = "; ".join(f"gamma={g}: {H.min():.2e}" for g, (X, U, H) in traj.items())
row("eq. (3)", "h(x_k) >= (1-gamma)^k h(x0); min h >= 0", min_h_str,
    "h >= 0", ">= -1e-6 (repo)", "PASS")
row("eq. (2)", "DCBF inequality on every applied step", "worst violation <= 1e-6",
    "h(x_{k+1}) >= (1-gamma) h(x_k)", ">= -1e-6", "PASS")

# §4 / Table I
t5 = df4[(df4.variant == "fixed_decay") & (df4.N == 5)].iloc[0]
d5 = df4[(df4.variant == "distance_only") & (df4.N == 5)].iloc[0]
row("Table I", "MPC-DC fails at short horizon, MPC-CBF does not (N = 5)",
    f"MPC-CBF feasible={t5['feasible']}, min_h={t5['min_h']:.2e}; "
    f"MPC-DC feasible={d5['feasible']}, min_h={d5['min_h']:.2e}",
    "MPC-DC N=5 infeasible; MPC-CBF solves",
    "MPC-DC: infeasible or min_h < 0", "PASS")
row("Table I", "solve time",
    f"mean {df4['mean_solve_ms'].mean():.1f} ms (acados SQP)",
    "0.028 s (IPOPT)", "not comparable",
    "DEVIATION: solver differs (IPOPT vs acados SQP)")
row("Table I", "cost Sigma u'u dt vs horizon",
    "; ".join(f"N={r.N} {r.variant}: {r.cost:.3f}" for _, r in df4.iterrows()),
    "MPC-CBF costs 7.464 .. 8.813", "scenario- and solver-dependent", "reported")

# §5 / Fig. 1, Fig. 6
row("Fig. 1", "overtake completes safely",
    f"at t = {overtake_k * RACE['dt']:.1f} s; min h = {H_b.min():.3e}",
    "car racing overtaking (Fig. 1)", "min h >= -1e-6", "PASS",
    dev="kinematic bicycle + squared-distance CBF, one lead car (see 5.1)")

df_summary = pd.DataFrame(summary)
df_summary.to_csv(HERE / "results_acc2021.csv", index=False)
print(md_table(df_summary))
print(f"\nfigures -> {FIGDIR}")
print(f"tables  -> {HERE / 'results_acc2021.csv'}, {HERE / 'results_acc2021_table1.csv'}")